## Vehicle Routing Problem using OR-Tools
### Import the required libraries

In [1]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
import matplotlib as mpl
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

### Load the problem data from file.
Load the CSV file and store it into a dataframe. Set the <code>coordinates</code> from the <code>x</code> and <code>y</code> columns. Set the <code>demands</code> from the <code>d</code> column.


In [3]:
dataset = pd.read_csv("tai150d.csv", index_col=0)
coordinates = dataset.loc[:, ["x", "y"]]
demands = dataset.d.values

#### Set the parameters: capacity, number of vehicles, and $N$.

In [5]:
capacity = 1874
n_vehicles = 14
N = coordinates.shape[0]

#### Compute the distance matrix using <code>squareform</code>. 
Alternatively, you can can use <code>cdist</code> from <code>scipy.spatialdistance</code> or <code>euclidean_distances</code> from <code>sklearn.metrics.pairwise</code>. Make sure that the distances are rounded off and converted as integers.

In [30]:
distances = squareform(pdist(coordinates, metric="euclidean"))
distances = np.round(distances * 1e4, decimals=0).astype(int)

### Instantiate a routing manager and a model.

In [31]:
manager = pywrapcp.RoutingIndexManager(N, n_vehicles, 0)
routing = pywrapcp.RoutingModel(manager)

### Define the distance callback.
This returns the distances between locations, and passes it to the solver. It also sets the arc costs, which define the cost of travel, to be the distances of the arcs.

In [32]:
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distances[from_node, to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)

### Define the demand callback. 
This links the demands from the nodes to the capacity constraints expected by the solver. 

In [33]:
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return demands[from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

### Add Constraints
#### Add dimensions related to arcs/weights.
The <code>AddDimensionWithVehicleCapacity</code> accumulates the quantities along the route while imposing the capacity limit. It accepts the index of the demand_callback function, the slack allowed per stop (0), the vehicle limits, a boolean for whether routes start at the depot, and the dimension name.

In [34]:
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,  
    [capacity,] * n_vehicles,  
    True,  
    'Capacity'
)

True

### Set the search strategies
#### Set a first solution strategy
Use the default routing search parameters. Use the <code>PATH_CHEAPEST_ARC</code> as the strategy to look for the first solution. 

In [39]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

#### Include Guided Local Search as a metaheuristic
Set another attribute in the search parameters using the <code>local_search_metaheuristic</code> object.

In [40]:
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)

search_parameters.time_limit.seconds = 30

### Set the objective function
The objective function minimises the total transportation cost. Thus, the <code>transit_callback_index</code> is passed to the <code>SetArcCostEvaluatorOfAllVehicles</code> function.

In [41]:
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

### Solve the model
#### Print the solution

In [42]:
def print_solution(data, manager, routing, solution):
    print(f"Objective: {solution.ObjectiveValue()}")
    max_route_distance = 0
    for vehicle_id in range(n_vehicles):
        if not routing.IsVehicleUsed(solution, vehicle_id):
            continue
        index = routing.Start(vehicle_id)
        plan_output = f"Route for vehicle {vehicle_id}:\n"
        route_distance = 0
        while not routing.IsEnd(index):
            plan_output += f" {manager.IndexToNode(index)} -> "
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(
                previous_index, index, vehicle_id
            )
        plan_output += f"{manager.IndexToNode(index)}\n"
        plan_output += f"Distance of the route: {route_distance}m\n"
        print(plan_output)
        max_route_distance = max(route_distance, max_route_distance)
    print(f"Maximum of the route distances: {max_route_distance}m")

In [43]:
solution = routing.SolveWithParameters(search_parameters)

if solution:
    print_solution(distances, manager, routing, solution)
else:
    print("No solution found !")

Objective: 31686402
Route for vehicle 0:
 0 ->  79 ->  74 ->  81 ->  72 ->  73 ->  76 ->  78 -> 0
Distance of the route: 659121m

Route for vehicle 1:
 0 ->  11 ->  5 ->  1 ->  55 ->  54 ->  58 ->  61 ->  65 ->  60 ->  149 ->  146 ->  145 ->  148 ->  143 -> 0
Distance of the route: 1185152m

Route for vehicle 2:
 0 ->  63 ->  38 ->  41 ->  122 ->  49 ->  46 ->  43 ->  42 ->  37 ->  33 ->  40 ->  96 ->  97 ->  150 -> 0
Distance of the route: 2625821m

Route for vehicle 3:
 0 ->  141 ->  138 ->  144 ->  50 -> 0
Distance of the route: 1646697m

Route for vehicle 4:
 0 ->  36 ->  48 ->  44 ->  132 ->  133 ->  130 ->  128 -> 0
Distance of the route: 2200718m

Route for vehicle 5:
 0 ->  147 ->  98 ->  94 ->  95 ->  59 -> 0
Distance of the route: 1424331m

Route for vehicle 6:
 0 ->  16 ->  15 ->  26 ->  118 ->  116 ->  114 ->  111 ->  127 ->  129 ->  52 -> 0
Distance of the route: 3147275m

Route for vehicle 7:
 0 ->  22 ->  27 ->  14 ->  20 ->  12 ->  28 ->  17 ->  21 ->  23 ->  13 ->  24 